
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

# UofT FASE ML Bootcamp
#### Friday June 12, 2026
####  Retrieval-Augmented Generation (RAG) - Lab 1, Day 5
#### Teaching team: Eldan Cohen, Alex Olson, Hriday Chheda
##### Lab author: Hriday Chheda

In this lab, you will build a Retrieval-Augmented Generation (RAG) system step by step using a collection of Paul Graham essays.

The goal is to understand what happens inside a RAG pipeline.

In particular, you will focus on:

1. Loading a collection of documents
2. Splitting documents into chunks
3. Converting chunks into vector embeddings
4. Computing similarity scores between a question and all chunks
5. Selecting the top-k most relevant chunks
6. Building the prompt/context given to a language model
7. Generating an answer using only the retrieved context
8. Studying how chunk size, overlap, and top-k affect retrieval quality


---



We start by installing and importing the required libraries.

In [1]:
! pip install -q llama-index
! pip install -q sentence-transformers
! pip install -q transformers[torch]
! pip install -q pandas numpy requests tqdm sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [2]:
import os
import re
import html
import textwrap
import requests
import numpy as np
import pandas as pd

from tqdm import tqdm
from sentence_transformers import SentenceTransformer

from transformers import T5Tokenizer, T5ForConditionalGeneration

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

# 1. What is RAG?

A standard language model answers directly from its parameters:

```text
Question -> Language Model -> Answer
```

A RAG system first retrieves relevant external information:

```text
Question -> Retriever -> Relevant document chunks -> Language Model -> Answer
```

The important idea is that the language model does not need to memorize every fact. Instead, it can be given relevant information at inference time.

In this lab, we will make every step visible.

# 2. Download a multi-document Paul Graham essay dataset

A RAG system is most interesting when there are multiple documents and the system must decide which pieces of information are relevant.

Paul Graham is a programmer, entrepreneur, investor, and essayist. He co-founded Viaweb and Y Combinator, and is known for essays about startups, programming, work, wealth, and ideas.

His essays are useful for RAG because they are readable, varied, and spread across multiple documents. This lets us test whether retrieval can find the right essay and the right passage for a question.

We will download Paul Graham essays and save each essay as a separate `.txt` file.

In [ ]:
DATA_DIR = "data/paul_graham_essays"
os.makedirs(DATA_DIR, exist_ok=True)

essay_urls = {
    "wealth": "https://www.paulgraham.com/wealth.html",
    "startupideas": "https://www.paulgraham.com/startupideas.html",
    "schlep": "https://www.paulgraham.com/schlep.html",
    "makersschedule": "https://www.paulgraham.com/makersschedule.html",
    "greatwork": "https://www.paulgraham.com/greatwork.html",
    "superlinear": "https://www.paulgraham.com/superlinear.html",
    "kids": "https://www.paulgraham.com/kids.html",
    "before": "https://www.paulgraham.com/before.html",
    "taste": "https://www.paulgraham.com/taste.html",
    "mean": "https://www.paulgraham.com/mean.html",
}

def clean_html_to_text(raw_html):
    """A small HTML cleaner so that we do not need BeautifulSoup."""
    raw_html = re.sub(r"<script.*?</script>", " ", raw_html, flags=re.DOTALL | re.IGNORECASE)
    raw_html = re.sub(r"<style.*?</style>", " ", raw_html, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", raw_html)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for essay_name, url in essay_urls.items():
    output_path = os.path.join(DATA_DIR, f"{essay_name}.txt")
    if os.path.exists(output_path):
        continue
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    text = clean_html_to_text(response.text)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(text)

print(f"Saved essays to: {DATA_DIR}")
print("Files:", sorted(os.listdir(DATA_DIR)))

## 2.1 Inspect the raw text files


In [ ]:
for filename in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, filename)
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    print("=" * 80)
    print(filename)
    print("Number of characters:", len(text))
    print("Preview:")
    print(textwrap.fill(text[:700], width=100))

## 2.2 Exercise

Choose one essay file and print a longer preview of it.

In [ ]:
# TODO: Change this filename to inspect a different essay.
chosen_file =

path = os.path.join(DATA_DIR, chosen_file)
with open(path, "r", encoding="utf-8") as f:
    text = f.read()

print(textwrap.fill(text[:2000], width=100))

# 3. Load the documents with LlamaIndex

LlamaIndex provides useful utilities for loading files and creating document objects.

At this stage, we are **not** building a RAG system yet. We are only loading documents.

In [ ]:
documents = SimpleDirectoryReader(DATA_DIR).load_data()

print(f"Number of loaded documents: {len(documents)}")
print(type(documents[0]))
print(documents[0].metadata)


Let us make a small table showing the metadata and size of each document.

In [ ]:
doc_rows = []

for i, document in enumerate(documents):
    doc_rows.append({
        "document_id": i,
        "file_name": document.metadata.get("file_name"),
        "num_characters": len(document.text),
        "preview": document.text[:120].replace("\n", " ")
    })

doc_df = pd.DataFrame(doc_rows)
doc_df

# 4. Chunking documents

Most RAG systems do not embed entire documents at once. Instead, documents are split into smaller pieces called **chunks** or **nodes**.

Why?

- Entire documents may be too long for an embedding model or language model.
- Retrieval works better when the system retrieves focused pieces of text.
- But if chunks are too small, they may lose important context.

This makes chunking a key design decision.

In [ ]:
def create_chunks(documents, chunk_size=256, chunk_overlap=15):
    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    nodes = splitter.get_nodes_from_documents(documents)
    return nodes

nodes_default = create_chunks(documents, chunk_size=256, chunk_overlap=15)

print(f"Number of documents: {len(documents)}")
print(f"Number of chunks/nodes: {len(nodes_default)}")
print(type(nodes_default[0]))

## 4.1 Inspect the chunks

Each chunk has:

- text
- a node ID
- metadata telling us which document it came from

In [ ]:
for i, node in enumerate(nodes_default[:5]):
    print("=" * 80)
    print(f"Chunk index: {i}")
    print("Node ID:", node.node_id)
    print("Source file:", node.metadata.get("file_name"))
    print("Number of characters:", len(node.text))
    print("Preview:")
    print(textwrap.fill(node.text[:700], width=100))

## 4.2 Exercise

Create chunks using a smaller chunk size and inspect the first few chunks.

What changes compared to the default chunking setup?

In [ ]:
# TODO: Try different values here.
small_chunk_size =
small_chunk_overlap =

nodes_small = create_chunks(
    documents,
    chunk_size=small_chunk_size,
    chunk_overlap=small_chunk_overlap
)

print(f"Number of small chunks: {len(nodes_small)}")

large_chunk_size =
large_chunk_overlap =

nodes_large = create_chunks(
    documents,
    chunk_size=large_chunk_size,
    chunk_overlap=large_chunk_overlap
)

print(f"Number of large chunks: {len(nodes_large)}")

# 5. Embeddings

A retriever needs a way to compare the user's question to all chunks.

To do this, we convert text into vectors called **embeddings**.

Texts with similar meanings should have embeddings that are close together.

We will use a small sentence-transformer model:

```text
sentence-transformers/all-MiniLM-L6-v2
```

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

Let us embed two short example sentences and compare their cosine similarity.

In [ ]:
example_texts = [
    "Startups should solve real problems.",
    "New companies need to address genuine customer needs.",
    "I like eating mangoes in the summer.",
    "Mangoes are my favorite fruit during hot weather."
]

example_embeddings = embedding_model.encode(
    example_texts,
    normalize_embeddings=True
)

similarity_matrix = example_embeddings @ example_embeddings.T

pd.DataFrame(
    similarity_matrix,
    index=example_texts,
    columns=example_texts
).round(3)

The cosine similarity is high when two vectors point in a similar direction.

# 6. Build a vector index

A vector index stores one embedding per chunk.

In this lab, we will build a very simple index ourselves using NumPy:

```text
chunks -> embedding model -> matrix of chunk embeddings
```

Then retrieval becomes:

```text
question embedding -> similarity with all chunk embeddings -> top-k chunks
```

In [ ]:
def build_vector_index(nodes, embedding_model):
    """Embed all chunks and return a simple dictionary-based vector index."""
    chunk_texts = [node.text for node in nodes]

    chunk_embeddings = embedding_model.encode(
        chunk_texts,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    index = {
        "nodes": nodes,
        "texts": chunk_texts,
        "embeddings": chunk_embeddings,
    }

    return index

In [ ]:
vector_index_default = build_vector_index(nodes_default, embedding_model)

print("Number of chunks:", len(vector_index_default["nodes"]))
print("Embedding matrix shape:", vector_index_default["embeddings"].shape)

The embedding matrix has shape:

```text
number of chunks x embedding dimension
```

Each row is one chunk embedding.

In [ ]:
# Inspect the first embedding vector.
first_embedding = vector_index_default["embeddings"][0]

print("Shape of one embedding:", first_embedding.shape)
print("First 10 values:", first_embedding[:10])

# 7. Retrieval

For any question, we will:

1. Embed the question
2. Compute similarity between the question embedding and every chunk embedding
3. Sort chunks by similarity score
4. Return the top-k chunks

In [ ]:
def retrieve_top_k(vector_index, question, embedding_model, top_k=3):
    """Retrieve the top-k chunks most similar to the question."""
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    chunk_embeddings = vector_index["embeddings"]

    # cosine similarity between question and each chunk (since embeddings are normalized, dot product = cosine similarity).
    similarity_scores = chunk_embeddings @ question_embedding

    # Sort by similarity score from highest to lowest.
    ranked_indices = np.argsort(similarity_scores)[::-1]
    top_indices = ranked_indices[:top_k]

    rows = []
    retrieved_nodes = []

    for rank, chunk_index in enumerate(top_indices, start=1):
        node = vector_index["nodes"][chunk_index]
        score = similarity_scores[chunk_index]

        rows.append({
            "rank": rank,
            "chunk_index": int(chunk_index),
            "similarity_score": float(score),
            "source_file": node.metadata.get("file_name"),
            "node_id": node.node_id,
            "text_preview": node.text[:350].replace("\n", " ")
        })

        retrieved_nodes.append(node)

    retrieval_df = pd.DataFrame(rows)

    return retrieval_df, retrieved_nodes, similarity_scores

## 7.1 Try one question

We will start with a factual question that should be answered by one of the essays.

In [ ]:
question = "How did Paul Graham study for exams?"

retrieval_df, retrieved_nodes, similarity_scores = retrieve_top_k(
    vector_index_default,
    question,
    embedding_model,
    top_k=5
)

retrieval_df

The table above is the retrieval step.

Notice that we have not used a language model yet. We only compared vector embeddings from the question and the different chunks

In [ ]:
print("Question:")
print(question)

for i, node in enumerate(retrieved_nodes, start=1):
    print("=" * 80)
    print(f"Retrieved chunk {i}")
    print("Source file:", node.metadata.get("file_name"))
    print(textwrap.fill(node.text[:1200], width=100))

## 7.2 Exercise

Try different questions and inspect which chunks are retrieved.

Suggested questions:

- What is the maker's schedule?
- What does Paul Graham mean by a schlep?
- What does Paul Graham say about doing great work?
- Why are startup ideas hard to find?
- How does Paul Graham define wealth?

In [ ]:
# TODO: Change this question.
question =

retrieval_df, retrieved_nodes, similarity_scores = retrieve_top_k(
    vector_index_default,
    question,
    embedding_model,
    top_k=5
)

retrieval_df

# 8. Build the context for the language model

A RAG system does not usually send the entire document collection to the language model.

It sends only the retrieved chunks.

We will now build the exact context that will be inserted into the prompt.

In [ ]:
def build_context(retrieved_nodes):
    context_blocks = []

    for i, node in enumerate(retrieved_nodes, start=1):
        block = f"""
[Chunk {i}]
Source file: {node.metadata.get('file_name')}

{node.text}
""".strip()
        context_blocks.append(block)

    context = "\n\n" + "=" * 80 + "\n\n"
    context = context.join(context_blocks)

    return context

context = build_context(retrieved_nodes)
print(context)

# 9. Load a language model for generation

Now we will use a small instruction-following model to generate an answer from the retrieved context.

We use Flan-T5 large

The model is not very powerful, but that is okay for this lab. The goal is to understand RAG mechanics.

In [ ]:
GENERATION_MODEL_NAME = "google/flan-t5-large"

generation_tokenizer = T5Tokenizer.from_pretrained(GENERATION_MODEL_NAME)
generation_model = T5ForConditionalGeneration.from_pretrained(GENERATION_MODEL_NAME)

Let us create a function that shows the exact prompt given to the model.

In [ ]:
def build_prompt(question, context):
    prompt = f"""
Answer the question using the context below. Answer in a complete sentence not just copy pasting the context.
If the answer is not contained in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:
""".strip()

    return prompt

In [ ]:
question = "How did Paul Graham study for exams?"
_, retrieved_nodes, _ = retrieve_top_k(
    vector_index_default,
    question,
    embedding_model,
    top_k=5
)

context = build_context(retrieved_nodes)
prompt = build_prompt(question, context)
print(prompt)

In [ ]:
def generate_answer(question, context, max_new_tokens=3500):
    prompt = build_prompt(question, context)

    input_ids = generation_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=False,
        max_length=1024
    ).input_ids

    outputs = generation_model.generate(
        input_ids,
        max_new_tokens=max_new_tokens
    )

    answer = generation_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [ ]:
question = "How did Paul Graham study for exams?"
retrieval_df, retrieved_nodes, similarity_scores = retrieve_top_k(
    vector_index_default,
    question,
    embedding_model,
    top_k=5
)
context = build_context(retrieved_nodes)
answer = generate_answer(question, context)
print(answer)

# 10. Put retrieval and generation together

Now we define a transparent RAG function.

This function returns three things:

1. the generated answer
2. a retrieval table showing which chunks were selected
3. the exact context sent to the language model

In [ ]:
def transparent_rag_answer(vector_index, question, embedding_model, top_k=3):
    retrieval_df, retrieved_nodes, similarity_scores = retrieve_top_k(
        vector_index,
        question,
        embedding_model,
        top_k=top_k
    )

    context = build_context(
        retrieved_nodes
    )

    answer = generate_answer(
        question,
        context
    )

    return answer, retrieval_df, context, similarity_scores

In [ ]:
question = "How did Paul Graham study for exams?"

answer, retrieval_df, context, similarity_scores = transparent_rag_answer(
    vector_index_default,
    question,
    embedding_model,
    top_k=5
)

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

retrieval_df

Try different questions

Suggested questions:

- What is the maker's schedule?
- What does Paul Graham mean by a schlep?
- What does Paul Graham say about doing great work?
- Why are startup ideas hard to find?
- How does Paul Graham define wealth?

In [ ]:
#TODO: Try different questions here.

# 11. Experiment 1: top-k retrieval

The parameter `top_k` controls how many chunks are retrieved.

- If `top_k` is too small, the relevant information may be missed.
- If `top_k` is too large, irrelevant context may distract the language model.

Let us compare `top_k = 1`, `top_k = 3`, and `top_k = 5`.

In [ ]:
question = "What does Paul Graham say about wealth?"

top_k_results = []

for top_k in [1, 3, 5]:
    answer, retrieval_df, context, similarity_scores = transparent_rag_answer(
        vector_index_default,
        question,
        embedding_model,
        top_k=top_k
    )

    top_k_results.append({
        "top_k": top_k,
        "answer": answer,
        "retrieved_files": list(retrieval_df["source_file"]),
        "top_score": retrieval_df.iloc[0]["similarity_score"]
    })

pd.DataFrame(top_k_results)

## 11.1 Exercise

For the table above, manually inspect the answers.

Questions to discuss:

1. Did increasing `top_k` improve the answer?
2. Did increasing `top_k` add irrelevant information?
3. Which retrieved source file seems most important?

# 12. Experiment 2: chunk size

Now we compare different chunking strategies.

Recall we built different chunking strategies in Section 4.2 apart from our default chunks we have being using so far:

1. Small chunks
3. Large chunks

We will ask the same question(s) and inspect the retrieved chunks.

In [ ]:
chunk_settings = [
    {"name": "small", "chunk_size": 50, "chunk_overlap": 15},
    {"name": "deafult", "chunk_size": 256, "chunk_overlap": 15},
    {"name": "large", "chunk_size": 750, "chunk_overlap": 15},
]

indexes_by_chunking = {}

for setting in chunk_settings:
    print("=" * 80)
    print(setting)

    nodes = create_chunks(
        documents,
        chunk_size=setting["chunk_size"],
        chunk_overlap=setting["chunk_overlap"]
    )

    vector_index = build_vector_index(nodes, embedding_model)

    indexes_by_chunking[setting["name"]] = {
        "setting": setting,
        "nodes": nodes,
        "index": vector_index
    }

    print("Number of chunks:", len(nodes))

In [ ]:
question = "How did Paul Graham study for exams?"

chunking_results = []

for name, item in indexes_by_chunking.items():
    answer, retrieval_df, context, similarity_scores = transparent_rag_answer(
        item["index"],
        question,
        embedding_model,
        top_k=5
    )

    chunking_results.append({
        "chunking": name,
        "chunk_size": item["setting"]["chunk_size"],
        "chunk_overlap": item["setting"]["chunk_overlap"],
        "num_chunks": len(item["nodes"]),
        "answer": answer,
        "retrieved_files": list(retrieval_df["source_file"]),
        "top_similarity_score": retrieval_df.iloc[0]["similarity_score"]
    })

pd.DataFrame(chunking_results)

## 12.1 Inspect retrieval for each chunking strategy

The answer table is useful, but the retrieval table is more important.

In [ ]:
question = "How did Paul Graham study for exams?"

for name, item in indexes_by_chunking.items():
    print("=" * 100)
    print("Chunking strategy:", name)

    retrieval_df, retrieved_nodes, similarity_scores = retrieve_top_k(
        item["index"],
        question,
        embedding_model,
        top_k=5
    )

    display(retrieval_df[["rank", "similarity_score", "source_file", "chunk_index", "text_preview"]])

## 12.2 Exercise

Try a different question and compare chunking strategies again.

In [ ]:
# TODO: Change this question and re-run the comparison.
question =

for name, item in indexes_by_chunking.items():
    print("=" * 100)
    print("Chunking strategy:", name)

    retrieval_df, retrieved_nodes, similarity_scores = retrieve_top_k(
        item["index"],
        question,
        embedding_model,
        top_k=3
    )

    display(retrieval_df[["rank", "similarity_score", "source_file", "chunk_index", "text_preview"]])

# 13. Experiment 4: Questions where RAG can fail

RAG can fail for several reasons:

1. The relevant document was not in the dataset.
2. The relevant chunk was not retrieved.
3. The chunk was retrieved, but the language model ignored it.
4. The question requires combining many pieces of information.
5. The wording of the question does not match the wording of the documents.

Let us test a question that may not be answerable from the essay collection.

In [ ]:
question = "What was Paul Graham's favourite food as a child?"

answer, retrieval_df, context, similarity_scores = transparent_rag_answer(
    vector_index_default,
    question,
    embedding_model,
    top_k=3
)

print("Question:")
print(question)
print("\nAnswer:")
print(answer)

retrieval_df

## 13.1 Exercise

Find one question where the RAG system gives a poor answer.

Then diagnose the failure:

- Was the answer absent from the documents?
- Did retrieval select the wrong chunks?
- Was the context too long or too short?
- Did the generator fail despite good retrieved context?

In [ ]:
# TODO: Write your own failure-case question.
question = ""

# TODO: Run transparent_rag_answer and inspect the retrieval table.

# 14. Summary

In this lab, you built a transparent RAG system from first principles.

You saw that a RAG system is made of several steps:

```text
Documents
    -> Chunks
    -> Embeddings
    -> Similarity scores
    -> Top-k retrieved chunks
    -> Context
    -> Prompt
    -> Generated answer
```

The key lesson is that RAG quality depends heavily on retrieval quality.

Changing chunk size, chunk overlap, top-k, and the question wording can all change the final answer.